# Шаг гуманоида: кинематическая симуляция на robolab v0.3

**Что моделируется.** Гуманоид на 21 степень свободы (~1.45 м, 37.5 кг) проходит 6 шагов по ровному полу.
1. **План походки.** Следы стоп, траектории переносимой ноги, опорная траектория ZMP.
2. **Траектория ЦМ** строится preview-регулятором Каджиты на модели «тележка на столе» (LIPM).
3. **Таз** подбирается так, чтобы *настоящий* ЦМ всего тела (ноги, руки) шёл по плану.
4. **IK ног** решается каждые 10 мс: метод наименьших квадратов с демпфированием на SE(3) (`SE3.log` + `SE3.adjoint`).
5. **Проверка баланса:** многозвенный ZMP по прямой кинематике всей траектории (один батч-вызов) должен лежать внутри опорного многоугольника.
6. **Монте-Карло:** куда уходит стопа из-за дискретности энкодеров 10–18 бит.
7. **Скорость:** OpenMP-ядро против наивного цикла.

**Допущения.** Это кинематика и критерий ZMP, а не физический движок: контакт идеальный, звенья считаются точечными массами (момент инерции звеньев не учитывается), моторы отрабатывают углы точно.

**Запуск:** Runtime → Run all, на шаге 1 загрузить `robolab-0.3.zip`.

## 1. Установка

In [ ]:
import glob, os
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not glob.glob("robolab*.zip"):
    if IN_COLAB:
        files.upload()                      # выберите robolab-0.3.zip
    else:
        raise FileNotFoundError("положите robolab-0.3.zip рядом с ноутбуком")
ZIP = sorted(glob.glob("robolab*.zip"))[-1]
print("архив:", ZIP)
!rm -rf robolab && unzip -oq {ZIP}
!pip -q install ./robolab             # Cython + OpenMP, ~40 с

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

import robolab.transforms as rt
from robolab.transforms import SE3, SO3, SE3Array
from robolab.humanoid import make_humanoid, simulate_walk, GaitParams, plan_feet, preview_control, LEG_JOINTS, SKELETON
from robolab.humanoid.analysis import encoder_quantization_mc

print("ядро:", rt.get_backend(), "| OpenMP:", rt.openmp_enabled(), "| потоков:", rt.get_num_threads(),
      "| CPU в системе:", os.cpu_count())
assert rt.get_backend() == "cython", "Cython-ядро не собралось — смотрите вывод pip"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 2. Модель

In [ ]:
robot = make_humanoid()
d = robot.dims
print(f"звеньев {robot.nl}, суставов {robot.nq}, масса {robot.total_mass:.1f} кг, длина ноги {d.leg_length:.2f} м")
pd.DataFrame({
    "сустав": robot.joint_names,
    "мин, °": np.degrees(robot.limits[:, 0]).round(0),
    "макс, °": np.degrees(robot.limits[:, 1]).round(0),
}).set_index("сустав").T

In [ ]:
SIDE_COLOR = {"l": "C2", "r": "C1"}          # левая — зелёная, правая — оранжевая

def draw_robot(ax, poses, alpha=1.0, lw=2.5):
    P = poses[:, :3, 3]
    for a, b in SKELETON:
        i, j = robot.link_id(a), robot.link_id(b)
        sides = {n[0] for n in (a, b) if n[:2] in ("l_", "r_")}
        color = SIDE_COLOR[sides.pop()] if len(sides) == 1 else "0.2"
        ax.plot(*zip(P[i], P[j]), color=color, alpha=alpha, lw=lw)
    h = poses[robot.link_id("head")]
    ax.scatter(*(h[:3, :3] @ [0.02, 0, 0.12] + h[:3, 3]), s=160, color="0.2", alpha=alpha)
    for side in "lr":
        s = poses[robot.link_id(f"{side}_sole")]
        c = np.array([[d.foot_length / 2, d.foot_width / 2, 0], [d.foot_length / 2, -d.foot_width / 2, 0],
                      [-d.foot_length / 2, -d.foot_width / 2, 0], [-d.foot_length / 2, d.foot_width / 2, 0],
                      [d.foot_length / 2, d.foot_width / 2, 0]]) @ s[:3, :3].T + s[:3, 3]
        ax.plot(*c.T, color=SIDE_COLOR[side], alpha=alpha, lw=1.5)

def equal_3d(ax, center, r):
    for set_lim, c in zip((ax.set_xlim, ax.set_ylim, ax.set_zlim), center):
        set_lim(c - r, c + r)
    ax.set_box_aspect((1, 1, 1))

q0 = robot.zero()
for s in "lr":
    q0[robot.joint_id(f"{s}_hip_pitch")], q0[robot.joint_id(f"{s}_knee")], q0[robot.joint_id(f"{s}_ankle_pitch")] = -0.35, 0.7, -0.35
pose0 = robot.fk(q0, SE3.Trans(0, 0, 0.68))
fig = plt.figure(figsize=(5, 6)); ax = fig.add_subplot(projection="3d")
draw_robot(ax, pose0.matrices)
c = robot.com(pose0); ax.scatter(*c, color="r", s=40, label=f"ЦМ, z={c[2]:.2f} м")
equal_3d(ax, (0, 0, 0.75), 0.8); ax.view_init(15, -60); ax.legend(); plt.show()

## 3. План походки и preview-регулятор

ZMP-опора (серое) переходит со стопы на стопу. Чтобы ZMP модели «тележка на столе» шёл за ней, ЦМ должен **заранее** уходить в сторону опорной ноги: регулятор смотрит на 1.6 с вперёд.

In [ ]:
params = GaitParams()            # шаг 0.20 м, период 0.70 с, 6 шагов
plan = plan_feet(params)
com_plan, _, zmp_cart = preview_control(plan.zmp_ref, params.dt, zc=0.80)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
for k, lab in enumerate("xy"):
    ax[k].plot(plan.t, plan.zmp_ref[:, k], color="0.6", lw=3, label="ZMP опорный")
    ax[k].plot(plan.t, zmp_cart[:, k], "C3--", label="ZMP модели")
    ax[k].plot(plan.t, com_plan[:, k], "C0", label="ЦМ (план)")
    ax[k].set_title(f"ось {lab}"); ax[k].set_xlabel("t, с"); ax[k].set_ylabel("м")
ax[0].legend(); plt.tight_layout(); plt.show()
print(f"ошибка слежения ZMP модели: {np.abs(zmp_cart - plan.zmp_ref).max()*1000:.1f} мм")

## 4. Симуляция

In [ ]:
t = time.perf_counter()
walk = simulate_walk(robot, params, com_passes=4)
print(f"симуляция {time.perf_counter()-t:.2f} с: {len(walk.t)} шагов по {params.dt*1000:.0f} мс")
print("время по этапам, с:", {k: round(v, 3) for k, v in walk.timings.items()})
print(f"IK: {walk.ik_iterations} итераций, неудач {walk.ik_failures}")
print("ошибка ЦМ всего тела по проходам коррекции таза, мм:", np.round(np.array(walk.com_iterations) * 1000, 2))

## 5. Баланс

ZMP по многозвенной модели учитывает то, что LIPM игнорирует: разгон переносимой ноги и рук. Запас — расстояние от ZMP до края опорного многоугольника (больше нуля — внутри).

In [ ]:
from robolab.humanoid.balance import foot_corners
fig, ax = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [1.4, 1]})
for side in "lr":                                           # следы стоп в моменты опоры
    corners = foot_corners(walk.poses[:, robot.link_id(f"{side}_sole")], d.foot_length, d.foot_width)
    on_ground = walk.feet_ref[side][:, 2] < 1e-9
    for k in np.flatnonzero(on_ground)[::25]:
        ax[0].fill(*corners[k].T, color=SIDE_COLOR[side], alpha=0.06, lw=0)
ax[0].plot(*walk.zmp_ref.T, color="0.5", lw=2.5, label="ZMP опорный")
ax[0].plot(*walk.zmp.T, "C3", lw=1.2, label="ZMP многозвенный")
ax[0].plot(*walk.com[:, :2].T, "C0", lw=1.5, label="ЦМ")
ax[0].set_aspect("equal"); ax[0].set_title("вид сверху: следы стоп (левая зелёная, правая оранжевая), ZMP, ЦМ")
ax[0].legend(loc="upper left"); ax[0].set_xlabel("x, м"); ax[0].set_ylabel("y, м")

ax[1].plot(walk.t, walk.margin * 1000, "k")
ax[1].axhline(0, color="r", lw=1)
for ph, col in ((1, "C2"), (2, "C1")):
    ax[1].fill_between(walk.t, -10, 120, where=walk.phase == ph, color=col, alpha=0.12, lw=0)
ax[1].set_ylim(-10, 120); ax[1].set_ylabel("запас ZMP, мм"); ax[1].set_xlabel("t, с")
ax[1].set_title("запас устойчивости (зелёным — опора на левую, оранжевым — на правую)")
plt.tight_layout(); plt.show()
print(f"мин. запас {walk.margin.min()*1000:.1f} мм, ZMP внутри {100*(walk.margin > 0).mean():.1f}% времени")
print(f"макс. отклонение многозвенного ZMP от опорного: {np.abs(walk.zmp - walk.zmp_ref).max(0).round(3)} м (x, y)")

Что будет, если считать, что ЦМ совпадает с тазом (без коррекции)?

In [ ]:
naive = simulate_walk(robot, params, com_passes=1)
pd.DataFrame({
    "ЦМ = таз":               [naive.com_iterations[-1] * 1000, naive.margin.min() * 1000],
    "с коррекцией таза":      [walk.com_iterations[-1] * 1000, walk.margin.min() * 1000],
}, index=["макс. ошибка ЦМ, мм", "мин. запас ZMP, мм"]).round(1)

## 6. Траектории суставов

Для выбора приводов важны диапазон и скорости.

In [ ]:
leg = LEG_JOINTS["l"]
ids = [robot.joint_id(j) for j in leg]
qd = np.gradient(walk.q, params.dt, axis=0)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for j, name in zip(ids, leg):
    ax[0].plot(walk.t, np.degrees(walk.q[:, j]), label=name[2:])
    ax[1].plot(walk.t, np.degrees(qd[:, j]), label=name[2:])
ax[0].set_title("углы левой ноги, °"); ax[1].set_title("скорости, °/с")
for a in ax: a.set_xlabel("t, с")
ax[0].legend(ncol=3, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

pd.DataFrame({
    "диапазон, °": [f"{np.degrees(walk.q[:, j].min()):.0f} … {np.degrees(walk.q[:, j].max()):.0f}" for j in ids],
    "макс |ω|, °/с": [np.degrees(np.abs(qd[:, j]).max()).round(0) for j in ids],
    "макс |ω|, об/мин": [(np.abs(qd[:, j]).max() * 60 / (2 * np.pi)).round(1) for j in ids],
}, index=leg)

## 7. Анимация

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

stride = 4                                    # каждый 4-й кадр → реальное время при 25 fps
frames = range(0, len(walk.t), stride)
fig = plt.figure(figsize=(7, 6.5)); ax = fig.add_subplot(projection="3d")

def update(k):
    ax.cla()
    draw_robot(ax, walk.poses[k])
    ax.plot(*np.c_[walk.zmp[:k + 1], np.zeros(k + 1)].T, "C3", lw=1)
    ax.scatter(*walk.zmp[k], 0, color="C3", s=25)
    ax.scatter(*walk.com[k, :2], 0, color="C0", s=25, marker="x")
    poly = np.vstack([walk.polygons[k], walk.polygons[k][:1]])
    ax.plot(poly[:, 0], poly[:, 1], 0, color="C2", lw=1.5)
    equal_3d(ax, (walk.com[k, 0], 0, 0.72), 0.72)
    ax.view_init(10, -55)
    ax.set_title(f"t = {walk.t[k]:.2f} с   запас ZMP {walk.margin[k]*1000:.0f} мм", fontsize=10)

anim = FuncAnimation(fig, update, frames=frames)
anim.save("walk.gif", writer=PillowWriter(fps=25), dpi=80)
plt.close(fig)
display(Image(filename="walk.gif"))

## 8. Где кончается применимость LIPM

Перебор шага и периода. План по модели «тележка на столе» всегда выполним (ZMP модели совпадает с опорным), но реальное тело с тяжёлыми ногами отклоняет ZMP. Таблица показывает минимальный запас многозвенного ZMP; отрицательное значение значит, что робот опрокинется.

In [ ]:
step_times = [0.4, 0.5, 0.6, 0.7, 0.8]
step_lengths = [0.15, 0.20, 0.25, 0.30]
grid = np.zeros((len(step_lengths), len(step_times)))
t = time.perf_counter()
for i, sl in enumerate(step_lengths):
    for j, st in enumerate(step_times):
        w = simulate_walk(robot, GaitParams(step_length=sl, step_time=st, n_steps=4), com_passes=3)
        grid[i, j] = w.margin.min() * 1000
print(f"{grid.size} симуляций за {time.perf_counter()-t:.0f} с")
df = pd.DataFrame(grid, index=[f"шаг {s:.2f} м" for s in step_lengths],
                  columns=[f"{s:.1f} с" for s in step_times])
display(df.style.format("{:.0f} мм").background_gradient(cmap="RdYlGn", vmin=-60, vmax=40))

## 9. Монте-Карло: разрядность энкодеров и постановка стопы

Поза в середине переноса левой ноги. Каждый из 21 энкодера ошибается равномерно в пределах ±½ LSB. Где окажется переносимая стопа **относительно опорной** (`T_rsole_lsole = T_world_rsole⁻¹ · T_world_lsole`, батч `SE3Array`) и насколько сместится ЦМ?

In [ ]:
k = int(np.argmax(walk.feet_ref["l"][:, 2]))           # верхняя точка переноса левой стопы
q_mid, base_mid = walk.q[k], walk.base[k]
rng = np.random.default_rng(0)
rows, samples = {}, {}
t = time.perf_counter()
for bits in (10, 12, 14, 16, 18):
    r = encoder_quantization_mc(robot, q_mid, base_mid, bits, 100_000, rng)
    e = np.linalg.norm(r["pos"], axis=1) * 1000
    samples[bits] = e
    rows[f"{bits} бит"] = {
        "LSB, °": np.degrees(r["lsb"]),
        "стопа: среднее, мм": e.mean(), "стопа: 95%, мм": np.percentile(e, 95), "стопа: макс, мм": e.max(),
        "ориентация 95%, °": np.degrees(np.percentile(r["ang"], 95)),
        "ЦМ 95%, мм": np.percentile(np.linalg.norm(r["com"], axis=1), 95) * 1000,
    }
print(f"5 × 100 000 конфигураций за {time.perf_counter()-t:.2f} с")
display(pd.DataFrame(rows).T.style.format("{:.3g}"))

fig, ax = plt.subplots(figsize=(9, 3.5))
for bits, e in samples.items():
    ax.hist(e, bins=np.logspace(-3.5, 1.2, 120), histtype="step", lw=1.5, label=f"{bits} бит")
ax.set_xscale("log"); ax.set_xlabel("ошибка положения переносимой стопы, мм"); ax.legend()
plt.show()

## 10. Скорость: прямая кинематика всего тела для 20 000 конфигураций

In [ ]:
import timeit

def fk_naive(q, base):
    # то, что пишут без батч-ядра: цикл по звеньям с объектами SE3
    out = []
    for lk in robot.links:
        T = (base if lk.parent is None else out[robot.link_id(lk.parent)]) @ lk.offset
        if lk.joint is not None:
            T = T @ SE3.from_rotation(SO3.exp(q[robot.joint_id(lk.joint)] * np.asarray(lk.axis)))
        out.append(T)
    return out

def per_call(f, number=3):
    return min(timeit.repeat(f, number=number, repeat=3)) / number

N = 20_000
Q = np.random.default_rng(1).uniform(-0.5, 0.5, (N, robot.nq))
base = SE3.Trans(0, 0, 0.68)
res = {}
res["цикл SE3-объектов"] = per_call(lambda: [fk_naive(q, base) for q in Q[:200]], 1) / 200 * N
rt.set_backend("python");  res["батч, NumPy-ядро"] = per_call(lambda: robot.fk_batch(Q, base), 1)
rt.set_backend("cython")
n_cpu = os.cpu_count()
rt.set_num_threads(1);     res["батч, Cython, 1 поток"] = per_call(lambda: robot.fk_batch(Q, base))
rt.set_num_threads(n_cpu); res[f"батч, Cython, {n_cpu} потока"] = per_call(lambda: robot.fk_batch(Q, base))

s = pd.Series(res) * 1000
display(pd.DataFrame({"время, мс": s.round(1), "ускорение": (s.iloc[0] / s).round(0).astype(int).astype(str) + "×",
                      "конфигураций/с": (N / s * 1000).round(-3).astype(int)}))
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.barh(s.index[::-1], s.values[::-1], color=["C2", "C2", "C1", "C3"][: len(s)])
ax.set_xscale("log"); ax.set_xlabel(f"мс на {N} конфигураций × {robot.nl} звеньев"); plt.show()

In [ ]:
from robolab.transforms import _core_cy as K
T = np.ascontiguousarray(K.se3_exp_batch(np.random.default_rng(2).normal(size=(200_000, 6))))
X = np.ascontiguousarray(np.random.default_rng(3).normal(size=(200_000, 6)))
rows = {}
for name, f in [("композиция", lambda: K.se3_mul_batch(T, T)), ("обратное", lambda: K.se3_inv_batch(T)),
                ("exp", lambda: K.se3_exp_batch(X)), ("log", lambda: K.se3_log_batch(T))]:
    rt.set_num_threads(1);     t1 = per_call(f)
    rt.set_num_threads(n_cpu); tn = per_call(f)
    rows[name] = {"1 поток, мс": t1 * 1000, f"{n_cpu} потока, мс": tn * 1000, "ускорение": t1 / tn}
print("батчевые ядра SE(3), 200 000 элементов")
pd.DataFrame(rows).T.round(2)

## Выводы, которые стоит проверить на своей модели

- **Коррекция таза обязательна.** ЦМ всего тела отличается от таза на сантиметры; без поправки ZMP-план не выполняется точно.
- **LIPM работает в ограниченной области.** При коротком периоде или длинном шаге разгон переносимой ноги выводит многозвенный ZMP за опору, хотя план по LIPM формально корректен. Таблица в разделе 8 показывает эту границу для данных масс и длин.
- **Энкодеры.** Ошибка постановки стопы растёт примерно вдвое на каждый потерянный бит. Таблица в разделе 9 даёт разрядность под требуемую точность.
- **Скорость.** Прямая кинематика всего тела одним батч-вызовом Cython+OpenMP быстрее цикла по объектам SE3 на 2–3 порядка. Именно это делает Монте-Карло и перебор параметров интерактивными. На Colab всего 2 ядра, поэтому OpenMP даёт там до ×2, а на рабочей машине с 8–16 ядрами — пропорционально больше.